# 3 — Reproduce the figures

The paper's two figures, from the prepared subsets and the stored prediction
records. Figure 1 needs the image subsets; Figure 2 needs the prediction records
written by `scripts/evaluate.py`.

Command-line equivalents, which are the exact invocations the published PNGs
came from:

```bash
python scripts/make_figures.py --crops-triptych --seed 7 --triptych-rows 3 \
    --histogram-pool all --triptych-suffix _poolall
python scripts/make_figures.py --reliability --arms unfrozen_no_aug_seed42 \
    --reliability-metrics rotation --font-scale 1.0
```

A third figure, rotation error per arm against the chance level, is at the
bottom. It is no longer in the paper, but it needs no data at all, which makes it
the cheapest way to see the headline numbers.

In [ ]:
# --- environment ------------------------------------------------------------
# Local: nothing to do beyond `pip install -e .` in the repo root.
# Colab: uncomment, then set the two roots below.
#
# from google.colab import drive; drive.mount("/content/drive")
# !pip install -q -e "/content/drive/MyDrive/<path to repo>"

import os

# Where the SPEED+ download and the prepared subsets live.
os.environ.setdefault("SPEEDPLUS_DATA_ROOT", "/path/to/speedplusv2")
# Where checkpoints, predictions and figures go. On Colab put this on the
# mounted Drive so a disconnect does not cost the run.
os.environ.setdefault("SPEEDPLUS_ARTIFACTS", "/path/to/artifacts")

# On Colab, unpack image archives to local disk rather than reading tens of
# thousands of small files across the Drive mount. Set to None when local.
LOCAL_IMAGES = None  # e.g. "/content/images"

In [ ]:
from pathlib import Path

OUTPUT = Path("figures")
OUTPUT.mkdir(exist_ok=True)

## Figure 1 — What the augmentation actually models

The figure that carries the argument. Three columns: augmented synthetic training
samples, then real test samples from each domain. Underneath each is that
column's pixel-intensity density on a log count axis, with the augmented density
overlaid on both real columns as a dashed line.

The augmented distribution is a poor model of **both** real domains, and a
markedly less poor one of the diffuse lightbox, with total variation 0.63 against
0.80. That states the mechanism claim as a measurement rather than an appeal to
the eye, and it is why augmentation improves one domain on every seed while
leaving the other inside seed noise.

Two things make it a measurement rather than an anecdote, and both are easy to
get wrong:

- The histograms pool over the domain rather than over the crops on display. At a
  pool of three, a column sits 0.12–0.35 TV from its own true distribution, the
  same order as the between-domain difference the figure exists to show. A pool
  of 500 is within 0.019 of the full split; the paper used the whole split.
- The strip panels share one y-axis. Left to autoscale independently on a log
  scale, the same reference density renders at a different height in each panel,
  and every cross-panel comparison becomes an artifact of the axis.

In [ ]:
import numpy as np
import torch

from speedplus_pose.config import AugmentationConfig, load_config
from speedplus_pose.experiment import build_dataset
from speedplus_pose.figures import crops_with_histograms
from speedplus_pose.paths import Paths
from speedplus_pose.transforms import build_train_transform

SEED = 7      # the seed the published figure was drawn with
ROWS = 3      # crops shown per column
POOL = 500    # images per column the histograms accumulate over.
              # None = the whole split, which is what the paper used.

paths = Paths()
config = load_config("unfrozen_aug")

# The crops are picked with a seeded numpy generator, but the augmentation stack
# draws from torch's *global* RNG. Seed both, or the same SEED picks the same
# base images and augments them differently on every run.
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# preview=True keeps the stack identical but skips normalisation, so it renders.
plain = build_train_transform(AugmentationConfig(enabled=False), config.image, preview=True)

datasets = {
    "augmented synthetic": build_dataset(
        "train", paths,
        build_train_transform(config.augmentation, config.image, preview=True),
        local_images=LOCAL_IMAGES,
    ),
    "lightbox": build_dataset("lightbox", paths, plain, local_images=LOCAL_IMAGES),
    "sunlamp": build_dataset("sunlamp", paths, plain, local_images=LOCAL_IMAGES),
}

columns = {
    label: [dataset[int(i)][0].numpy()
            for i in rng.choice(len(dataset), ROWS, replace=False)]
    for label, dataset in datasets.items()
}

In [ ]:
def histogram_pool(dataset, size, seed=SEED):
    """One column's histogram pool, as a generator.

    A generator rather than a list on purpose: crops_with_histograms keeps only
    bin counts, so a 28,000-image pool costs one image of memory instead of the
    ~37 GB the pixels would take. size=None means the whole split.
    """
    if size is None:
        indices = range(len(dataset))
    else:
        indices = np.random.default_rng(seed).choice(
            len(dataset), min(size, len(dataset)), replace=False
        )
    return (dataset[int(i)][0].numpy() for i in indices)


crops_with_histograms(
    columns,
    path=OUTPUT / "crops_triptych.png",
    reference="augmented synthetic",
    pools={label: histogram_pool(dataset, POOL)
           for label, dataset in datasets.items()},
)

## Figure 2 — Conformal reliability

Nominal against measured coverage for the attitude cone, swept across the whole
range, for the layer-4 no-augmentation arm. The diagonal is a perfectly honest
model, and distance below it is how much the stated confidence overstates
reality. In-domain coverage tracks the diagonal, and both real domains collapse
far below it. The guarantee is intact; what broke is exchangeability.

The paper's panel is rotation only, at `font_scale=1.0`. At 1.4 the x-axis label
is clipped by `tight_layout`, and a single panel is already twice as wide as each
of the two it replaced.

The title is off by default, since it names the seed and the paper's build is
anonymised.

In [ ]:
from speedplus_pose.coverage import scores_from_record, sweep_reliability
from speedplus_pose.evaluate import load_predictions
from speedplus_pose.figures import reliability

ARM = "unfrozen_no_aug_seed42"     # the arm the paper's Figure 2 shows
PLOT_DOMAINS = ("in_domain_test", "sunlamp", "lightbox")

alphas = np.round(np.arange(0.01, 0.51, 0.01), 2)
nominal = 1 - alphas

records = load_predictions(paths.predictions / f"{ARM}.pkl")
scores = {s: scores_from_record(records[s]) for s in ("calibration", *PLOT_DOMAINS)}
curves = {
    "rotation": sweep_reliability(
        scores["calibration"], {d: scores[d] for d in PLOT_DOMAINS}, "rotation", alphas
    )
}

reliability(nominal, curves, ARM, metrics=("rotation",), font_scale=1.0,
            path=OUTPUT / f"reliability_{ARM}_rotation.png")

`sweep_reliability` asserts that every curve is non-decreasing in nominal
coverage. The maths guarantees that, which is what makes it a sharp check on the
code: if it raises, the bug is in the pipeline rather than the data.

Add `"translation_metres"` to `curves` and to `metrics` for the two-panel
version, which is what earlier drafts carried.

## Extra — rotation error per arm and domain

Not in the paper. This was Figure 1 until the triptych displaced it, and every
number in it is already in Table 1. It is kept because it reads
`results/results_2x2.csv` and needs no data, no checkpoint and no GPU, which
makes it the fastest way to see the shape of the result.

The dashed line carries the figure. Read distances *from* it: an error of 2.05
rad against a 2.207 chance level is a model that has learned almost nothing about
orientation, which the raw number hides completely.

In [ ]:
import pandas as pd

from speedplus_pose.figures import rotation_error_by_arm

RESULTS = Path("..") / "results" / "results_2x2.csv"
LABELS = {"frozen_no_aug": "frozen", "frozen_aug": "frozen+aug",
          "unfrozen_no_aug": "layer 4", "unfrozen_aug": "layer 4+aug"}

results = pd.read_csv(RESULTS)
arms = list(LABELS)

errors, stds = {}, {}
for domain in ("synthetic", "lightbox", "sunlamp"):
    grouped = results[results["domain"] == domain].groupby("arm")["rotation"]
    errors[domain] = {LABELS[a]: grouped.mean()[a] for a in arms}
    stds[domain] = {LABELS[a]: (grouped.std(ddof=1).get(a, 0.0) or 0.0) for a in arms}

rotation_error_by_arm([LABELS[a] for a in arms], errors, stds,
                      OUTPUT / "rotation_error_by_arm.png")